In [1]:
# Hyper-parameter tunning

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings("ignore")

churn_df = pd.read_csv('Data/telecom_churn.csv')

In [3]:
churn_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
churn_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Parameters: we do not set this, it is learned by the algorithm through the modeling process

In [18]:
# Convert TotalCharges to numeric
churn_df["TotalCharges"] = pd.to_numeric(churn_df["TotalCharges"], errors="coerce")
churn_df = churn_df.dropna(subset=["TotalCharges"])

# Drop customerID as it's not a feature
churn_df = churn_df.drop("customerID", axis=1, errors="ignore")

# Separate target variable
y = churn_df["Churn"].map({"Yes": 1, "No": 0}).astype(int)
X = churn_df.drop("Churn", axis=1)

# One-hot encode all categorical columns
X = pd.get_dummies(X, drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [19]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
lr.coef_

#coef_ contains the important information about coefficients on our variables in the model. 
#We do not set this, it is learned by the algorithm through the modeling process

array([[ 2.60280085e-01, -6.05923173e-02,  5.38248705e-03,
         2.92781944e-04, -2.82725267e-03,  1.85196025e-02,
        -2.52933618e-01, -4.75377708e-01,  2.95110141e-01,
         2.71064240e-01,  5.81239834e-01, -1.18743615e-01,
        -1.18743615e-01, -6.11507629e-01, -1.18743615e-01,
        -2.62231692e-01, -1.18743615e-01, -3.80515724e-02,
        -1.18743615e-01, -5.81162430e-01, -1.18743615e-01,
         2.06818656e-01, -1.18743615e-01,  1.79800643e-01,
        -5.06641407e-01, -7.04998646e-01,  3.62566225e-01,
        -6.20408933e-02,  2.27766383e-01,  3.67847177e-03]])

In [21]:
# Create a list of original variable names from the training DataFrame
original_variables = X_train.columns

# Extract the coefficients of the logistic regression estimator
model_coefficients = lr.coef_[0]

# Create a dataframe of the variables and coefficients & print it out
coefficient_df = pd.DataFrame({"Variable" : original_variables, "Coefficient": model_coefficients})
print(coefficient_df)

                                 Variable  Coefficient
0                           SeniorCitizen     0.260280
1                                  tenure    -0.060592
2                          MonthlyCharges     0.005382
3                            TotalCharges     0.000293
4                             gender_Male    -0.002827
5                             Partner_Yes     0.018520
6                          Dependents_Yes    -0.252934
7                        PhoneService_Yes    -0.475378
8          MultipleLines_No phone service     0.295110
9                       MultipleLines_Yes     0.271064
10            InternetService_Fiber optic     0.581240
11                     InternetService_No    -0.118744
12     OnlineSecurity_No internet service    -0.118744
13                     OnlineSecurity_Yes    -0.611508
14       OnlineBackup_No internet service    -0.118744
15                       OnlineBackup_Yes    -0.262232
16   DeviceProtection_No internet service    -0.118744
17        

We have succesfully extracted and reviewed a very important parameter for the Logistic Regression Model. The coefficients of the model allow us to see which variables are having a larger or smaller impact on the outcome. Additionally the sign lets us know if it is a positive or negative relationship.

In [22]:
# Print out the top 3 positive variables
top_three_df = coefficient_df.sort_values(by="Coefficient", axis=0, ascending=False)[0:3]
print(top_three_df)

                          Variable  Coefficient
10     InternetService_Fiber optic     0.581240
26            PaperlessBilling_Yes     0.362566
8   MultipleLines_No phone service     0.295110
